# OTTO: Word2Vec + XGBoost GPU Ranker (End-to-End Pipeline)


This notebook combines **Word2Vec**-based candidate generation with an **XGBoost Ranker (GPU)** for an end-to-end OTTO recommendation pipeline:

1. Load and preprocess OTTO data (train/test).
2. Train **Word2Vec** on session item sequences.
3. Generate **top-K candidates** per session via item-item similarity.
4. Build **ranking features** for `(session, candidate_aid)` pairs.
5. Train **XGBoost Ranker (rank:pairwise)** with **graded labels** (orders=3, carts=2, clicks=1).
6. Batch **inference** and assemble a **Kaggle-ready submission**.
7. (Optional) Switch between CPU/GPU with a single flag.

> Works on Kaggle directly. Assumes the OTTO dataset is available as the competition dataset.


In [ ]:

# ======================
# Config & Environment
# ======================
import os, gc, sys, json, math, time, random
import numpy as np
import pandas as pd

# Word2Vec (gensim)
from gensim.models import Word2Vec

# XGBoost Ranker
import xgboost as xgb

# Paths (Kaggle default; falls back to local for testing)
KAGGLE_INPUT = "/kaggle/input/otto-recommender-system"
KAGGLE_WORK  = "/kaggle/working"
LOCAL_INPUT  = "./input/otto-recommender-system"

DATA_DIR = KAGGLE_INPUT if os.path.exists(KAGGLE_INPUT) else LOCAL_INPUT
OUT_DIR  = KAGGLE_WORK if os.path.exists(KAGGLE_WORK) else "./output"
os.makedirs(OUT_DIR, exist_ok=True)

# Fast parameters (tune for better score vs. runtime)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Candidate & ranking parameters
W2V_VECTOR_SIZE = 64
W2V_WINDOW = 20
W2V_MIN_COUNT = 2
W2V_EPOCHS = 5

RECENT_N = 30          # take last N interactions per session for candidate seeding
CAND_K_PER_RECENT = 20 # top-K similar items per seed item
MAX_CANDIDATES = 120   # cap merged candidates per session

# XGBoost
USE_GPU = True
XGB_PARAMS = {
    "objective": "rank:pairwise",
    "max_depth": 6,
    "eta": 0.12,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "tree_method": "gpu_hist" if USE_GPU else "hist",
    "random_state": SEED,
    "n_estimators": 600,
    "learning_rate": 0.08,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
}


In [ ]:

# ===============
# Helper Functions
# ===============
from collections import defaultdict, Counter

def read_parquet_safely(path, columns=None):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_parquet(path, columns=columns)

def load_train_events(data_dir):
    """Load OTTO train parquet files and return a single DataFrame with [session, aid, ts, type]."""
    train_dir = os.path.join(data_dir, "train_parquet")
    if not os.path.exists(train_dir):
        train_dir = data_dir
    paths = []
    for root, _, files in os.walk(train_dir):
        for f in files:
            if f.endswith(".parquet") and "train" in root.lower():
                paths.append(os.path.join(root, f))
            elif f.endswith(".parquet") and "train" in f.lower():
                paths.append(os.path.join(root, f))
    if not paths:
        for root, _, files in os.walk(data_dir):
            for f in files:
                if f.endswith(".parquet"):
                    paths.append(os.path.join(root, f))
    dfs = []
    for p in sorted(paths):
        try:
            df = read_parquet_safely(p, columns=["session", "aid", "ts", "type"])
            dfs.append(df)
        except Exception:
            df = read_parquet_safely(p)
            expected = ["session","aid","ts","type"]
            assert all(c in df.columns for c in expected), f"Unexpected columns in {p}: {df.columns}"
            dfs.append(df[expected])
    if not dfs:
        raise RuntimeError("No train parquet files found.")
    train = pd.concat(dfs, ignore_index=True)
    train["type"] = train["type"].astype("int8")
    return train

def load_test_events(data_dir):
    """Load OTTO test parquet files and return a single DataFrame with [session, aid, ts, type]."""
    test_dir = os.path.join(data_dir, "test_parquet")
    if not os.path.exists(test_dir):
        test_dir = data_dir
    paths = []
    for root, _, files in os.walk(test_dir):
        for f in files:
            if f.endswith(".parquet") and "test" in root.lower():
                paths.append(os.path.join(root, f))
            elif f.endswith(".parquet") and "test" in f.lower():
                paths.append(os.path.join(root, f))
    if not paths:
        raise RuntimeError("No test parquet files found.")
    dfs = []
    for p in sorted(paths):
        try:
            df = read_parquet_safely(p, columns=["session", "aid", "ts", "type"])
            dfs.append(df)
        except Exception:
            df = read_parquet_safely(p)
            expected = ["session","aid","ts","type"]
            assert all(c in df.columns for c in expected), f"Unexpected columns in {p}: {df.columns}"
            dfs.append(df[expected])
    test = pd.concat(dfs, ignore_index=True)
    test["type"] = test["type"].astype("int8")
    return test

def to_session_sentences(df):
    """Group by session and create sequences (sentences) of aids sorted by ts."""
    g = df.sort_values(["session", "ts"]).groupby("session")["aid"].apply(list)
    sentences = [[str(a) for a in lst] for lst in g.tolist()]
    return sentences

def build_word2vec(sentences, vector_size=64, window=20, min_count=2, epochs=5, seed=42):
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,
        workers=os.cpu_count(),
        seed=seed
    )
    model.train(sentences, total_examples=len(sentences), epochs=epochs)
    return model

def similar_items(model, aid, topn=20):
    try:
        return model.wv.most_similar(str(aid), topn=topn)
    except KeyError:
        return []

def make_candidates_for_session(model, history_aids, topk_per_item=20, cap=120):
    seeds = history_aids[-RECENT_N:] if len(history_aids) > RECENT_N else history_aids
    cand_scores = defaultdict(float)
    for i, a in enumerate(reversed(seeds)):
        w = 1.0 / (1 + i)
        sims = similar_items(model, a, topn=topk_per_item)
        for b, s in sims:
            cand_scores[int(b)] += s * w
    for j, a in enumerate(reversed(history_aids)):
        cand_scores[int(a)] += 0.05 / (1 + j)
    cands = sorted(cand_scores.items(), key=lambda x: -x[1])[:cap]
    return cands

def grade_label(event_types):
    if 2 in event_types:
        return 3
    if 1 in event_types:
        return 2
    if 0 in event_types:
        return 1
    return 0

def build_training_pairs(train_df, model, max_per_session=120):
    sess_grp = train_df.sort_values(["session","ts"]).groupby("session")
    truth_map = sess_grp.apply(lambda g: g.groupby("aid")["type"].apply(lambda s: set(s.values)).to_dict())

    rows = []
    start = time.time()
    for idx, (sess, g) in enumerate(sess_grp):
        if idx % 20000 == 0 and idx > 0:
            print(f"[build_training_pairs] processed {idx} sessions in {time.time()-start:.1f}s ...")
        history = g["aid"].tolist()
        cands = make_candidates_for_session(model, history, topk_per_item=CAND_K_PER_RECENT, cap=max_per_session)
        pos_aids = list(set(g["aid"].tolist()))
        if pos_aids:
            cset = set(a for a,_ in cands)
            for a in pos_aids:
                if a not in cset:
                    cands.append((a, 0.01))
        seen = set()
        final = []
        for a, s in sorted(cands, key=lambda x: -x[1]):
            if a in seen:
                continue
            seen.add(a)
            final.append((a, s))
            if len(final) >= max_per_session:
                break
        tmap = truth_map.loc[sess]
        for aid, w2v_score in final:
            etypes = tmap.get(aid, set())
            rel = grade_label(etypes)
            freq = history.count(aid)
            last_pos = len(history) - 1 - history[::-1].index(aid) if aid in history else -1
            recency_feat = (len(history) - last_pos) if last_pos >= 0 else 0
            rows.append((sess, aid, w2v_score, freq, recency_feat, rel))
    df_pairs = pd.DataFrame(rows, columns=["session","aid","w2v_sim","hist_freq","recency","relevance"])
    return df_pairs

def build_test_pairs(test_df, model, max_per_session=120):
    sess_grp = test_df.sort_values(["session","ts"]).groupby("session")
    rows = []
    start = time.time()
    for idx, (sess, g) in enumerate(sess_grp):
        if idx % 20000 == 0 and idx > 0:
            print(f"[build_test_pairs] processed {idx} sessions in {time.time()-start:.1f}s ...")
        history = g["aid"].tolist()
        cands = make_candidates_for_session(model, history, topk_per_item=CAND_K_PER_RECENT, cap=max_per_session)
        seen = set()
        final = []
        for a, s in sorted(cands, key=lambda x: -x[1]):
            if a in seen:
                continue
            seen.add(a)
            final.append((a, s))
            if len(final) >= max_per_session:
                break
        from collections import Counter
        freq_counter = Counter(history)
        last_pos_map = {}
        for i, a in enumerate(history):
            last_pos_map[a] = i
        for aid, w2v_score in final:
            freq = freq_counter.get(aid, 0)
            last_pos = last_pos_map.get(aid, -1)
            recency_feat = (len(history) - last_pos) if last_pos >= 0 else 0
            rows.append((sess, aid, w2v_score, freq, recency_feat))
    df_pairs = pd.DataFrame(rows, columns=["session","aid","w2v_sim","hist_freq","recency"])
    return df_pairs

def dmatrix_from_pairs(df_pairs, label_col=None, group_col="session"):
    features = df_pairs[["w2v_sim","hist_freq","recency"]].astype("float32")
    group_sizes = df_pairs.groupby(group_col).size().tolist()
    if label_col is not None:
        dtrain = xgb.DMatrix(features, label=df_pairs[label_col].astype("float32"))
        dtrain.set_group(group_sizes)
        return dtrain, group_sizes
    dtest = xgb.DMatrix(features)
    dtest.set_group(group_sizes)
    return dtest, group_sizes


In [ ]:

# ======================
# 1) Load OTTO Data
# ======================
print("DATA_DIR:", DATA_DIR)
train = load_train_events(DATA_DIR)
test  = load_test_events(DATA_DIR)

print("train shape:", train.shape, "test shape:", test.shape)
print("train head:"); display(train.head())
print("test head:"); display(test.head())

train["session"] = train["session"].astype(np.int64)
train["aid"] = train["aid"].astype(np.int64)
test["session"] = test["session"].astype(np.int64)
test["aid"] = test["aid"].astype(np.int64)

for c in ["session","aid"]:
    train[c] = pd.to_numeric(train[c], downcast="unsigned")
    test[c]  = pd.to_numeric(test[c],  downcast="unsigned")
gc.collect();


In [ ]:

# ======================
# 2) Train Word2Vec
# ======================
print("Building sentences for Word2Vec...")
sentences = to_session_sentences(train)

print("Training Word2Vec...")
w2v = build_word2vec(
    sentences,
    vector_size=W2V_VECTOR_SIZE,
    window=W2V_WINDOW,
    min_count=W2V_MIN_COUNT,
    epochs=W2V_EPOCHS,
    seed=SEED
)
w2v_path = os.path.join(OUT_DIR, "otto_w2v.model")
w2v.save(w2v_path)
print("Word2Vec saved to:", w2v_path)


In [ ]:

# ======================
# 3) Build Training Pairs
# ======================
print("Building training (session, candidate) pairs + labels ...")
train_pairs = build_training_pairs(train, w2v, max_per_session=MAX_CANDIDATES)
print("train_pairs shape:", train_pairs.shape)
display(train_pairs.head())

train_pairs = train_pairs[train_pairs["relevance"] > 0].reset_index(drop=True)
print("Filtered train_pairs shape:", train_pairs.shape)
display(train_pairs["relevance"].value_counts())


In [ ]:

# ======================
# 4) Train XGBoost Ranker
# ======================
print("Preparing DMatrix...")
dtrain, group = dmatrix_from_pairs(train_pairs, label_col="relevance", group_col="session")

print("Training XGBoost Ranker...")
evals_result = {}
ranker = xgb.train(
    params=XGB_PARAMS,
    dtrain=dtrain,
    num_boost_round=XGB_PARAMS.get("n_estimators", 600),
    evals=[(dtrain, "train")],
    evals_result=evals_result,
    verbose_eval=50
)
model_path = os.path.join(OUT_DIR, "xgb_ranker.json")
ranker.save_model(model_path)
print("XGBoost ranker saved to:", model_path)


In [ ]:

# ======================
# 5) Build Test Pairs & Inference
# ======================
print("Building test (session, candidate) pairs ...")
test_pairs = build_test_pairs(test, w2v, max_per_session=MAX_CANDIDATES)
print("test_pairs shape:", test_pairs.shape)
display(test_pairs.head())

print("Making predictions (batch)...")
dtest, group_test = dmatrix_from_pairs(test_pairs, label_col=None, group_col="session")
preds = ranker.predict(dtest)

test_pairs["score"] = preds


In [ ]:

# ======================
# 6) Assemble Kaggle Submission
# ======================
def top20_labels(df):
    return " ".join(map(str, df.sort_values("score", ascending=False)["aid"].head(20).tolist()))

print("Ranking within each session...")
ranked = test_pairs.sort_values(["session","score"], ascending=[True, False])
labels_by_session = ranked.groupby("session").apply(top20_labels)
labels_by_session = labels_by_session.reindex(sorted(labels_by_session.index))

sub_rows = []
for sess, labs in labels_by_session.items():
    sub_rows.append((f"{sess}_clicks", labs))
    sub_rows.append((f"{sess}_carts",  labs))
    sub_rows.append((f"{sess}_orders", labs))

submission = pd.DataFrame(sub_rows, columns=["session_type","labels"])
sub_path = os.path.join(OUT_DIR, "submission.csv")
submission.to_csv(sub_path, index=False)
print("Saved submission to:", sub_path)
display(submission.head(9))



## Notes & Tips

- **Speed vs Score**: This notebook prioritizes a strong baseline with moderate runtime. For better leaderboard scores, enrich features (co-visitation counts, time-weighted frequencies, item-level meta, etc.) and/or train **three rankers** (clicks/carts/orders).
- **Memory**: If you hit memory limits, reduce `RECENT_N`, `CAND_K_PER_RECENT`, and `MAX_CANDIDATES`, or train Word2Vec with a smaller `vector_size`.
- **GPU**: If you don't have a GPU, set `USE_GPU=False`. The code will switch to `hist` automatically.
- **Graded labels**: We use `orders=3 > carts=2 > clicks=1` within sessions for pairwise ranking. You can adjust the mapping.
- **Batch inference**: Prediction happens in a single `xgb.DMatrix` call, which is much faster than per-session loops.
